# **Assignment 3**

## **Task 0**

In [13]:
#a
!pip install ollama

import ollama

from ollama import chat
from ollama import ChatResponse

#b
response: ChatResponse = chat(model='gemma3:270m', messages=[
  {
    'role': 'user',
    'content': 'Why is the sky blue? Keep it short.',
  },
])

print(response.message.content)

The sky is blue because of a phenomenon called Rayleigh scattering. Sunlight is made up of all the colors of the rainbow. When sunlight hits the Earth's atmosphere, it interacts with these colors. The blue and violet light have shorter wavelengths than the other colors, so they are scattered more effectively by the air molecules. This scattering causes the sky to appear blue.



In [14]:
#c
response: ChatResponse = chat(model='gemma3:4b', messages=[
  {
    'role': 'user',
    'content': 'Why are sunsets red? Keep it short.',
  },
])

print(response.message.content)

Sunsets are red because of a phenomenon called Rayleigh scattering. When sunlight hits the atmosphere, shorter wavelengths (like blue and violet) are scattered away, leaving the longer wavelengths (red and orange) to reach our eyes directly. 

Essentially, the atmosphere is filtering out the blue light, leaving us with a red sunset!


## **Task 1**

In [30]:
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/sippohippo/PythonForBusinessInt2026/refs/heads/main/assignments/data/emails.csv", sep=";")

#a
def classify_email(headline):
    headline = headline.lower()
    spam_keywords = ["won", "free", "claim", "hot", "singles", "inheritance", "transfer", "urgent"]
    work_keywords = ["report", "project", "meeting", "manuscript", "performance", "review"]

    if any(word in headline for word in spam_keywords):
        return "spam"
    elif any(word in headline for word in work_keywords):
        return "work"
    else:
        return "unknown"

df["classification"] = df["headline"].apply(classify_email)

display(df)

,headline,classification
0,URGENT: Your account will be suspended within ...,spam
1,Congratulations! You have won a 1000€ gift car...,spam
2,Hot singles in your area are waiting to meet y...,spam
3,Re: Inheritance transfer of 4.5M USD pending y...,spam
4,Meeting agenda for Thursday's project review,work
5,"Q3 budget report attached, please review by Fr...",work
6,Reminder: Annual performance review scheduled ...,work
7,"Updated draft of the manuscript, comments welcome",work
8,Quick question about last week,unknown
9,Following up,unknown


In [27]:
#b Using gemma3:270m
def classify_email_llm(headline, model):
    response = ollama.chat(model=model, messages=[
        {"role": "user", "content": (
            f"Classify this email headline as exactly one of: spam, work, or unknown. "
            f"Use these keywords as guidance:\n"
            f"spam: won, free, claim, hot, singles, inheritance, transfer, urgent\n"
            f"work: report, project, meeting, manuscript, performance, review\n"
            f"If you can't decide between spam and work, return unknown.\n"
            f"Reply with only the single word classification, nothing else.\n\n"
            f"Headline: {headline}"
        )}
    ])
    return response["message"]["content"].strip().lower()

df["gemma3_270m"] = df["headline"].apply(lambda x: classify_email_llm(x, "gemma3:270m"))

display(df[["headline", "classification", "gemma3_270m"]])

,headline,classification,gemma3_270m
0,URGENT: Your account will be suspended within ...,spam,work
1,Congratulations! You have won a 1000€ gift car...,spam,spam
2,Hot singles in your area are waiting to meet y...,spam,unknown
3,Re: Inheritance transfer of 4.5M USD pending y...,spam,work
4,Meeting agenda for Thursday's project review,work,unknown
5,"Q3 budget report attached, please review by Fr...",work,spam
6,Reminder: Annual performance review scheduled ...,work,spam
7,"Updated draft of the manuscript, comments welcome",work,spam
8,Quick question about last week,unknown,spam
9,Following up,unknown,work


In [28]:
#c Using gemma3:4b
df["gemma3_4b"] = df["headline"].apply(lambda x: classify_email_llm(x, "gemma3:4b"))

display(df[["headline", "classification", "gemma3_270m", "gemma3_4b"]])

,headline,classification,gemma3_270m,gemma3_4b
0,URGENT: Your account will be suspended within ...,spam,work,spam
1,Congratulations! You have won a 1000€ gift car...,spam,spam,spam
2,Hot singles in your area are waiting to meet y...,spam,unknown,spam
3,Re: Inheritance transfer of 4.5M USD pending y...,spam,work,spam
4,Meeting agenda for Thursday's project review,work,unknown,work
5,"Q3 budget report attached, please review by Fr...",work,spam,work
6,Reminder: Annual performance review scheduled ...,work,spam,work
7,"Updated draft of the manuscript, comments welcome",work,spam,work
8,Quick question about last week,unknown,spam,unknown
9,Following up,unknown,work,unknown


In [29]:
#d Repeat both models 3 times and store results
results = df[["headline"]].copy()

for i in range(1, 4):
    results[f"270m_run{i}"] = df["headline"].apply(lambda x: classify_email_llm(x, "gemma3:270m"))
    results[f"4b_run{i}"]   = df["headline"].apply(lambda x: classify_email_llm(x, "gemma3:4b"))

display(results)

,headline,270m_run1,4b_run1,270m_run2,4b_run2,270m_run3,4b_run3
0,URGENT: Your account will be suspended within ...,spam,spam,spam,spam,work,spam
1,Congratulations! You have won a 1000€ gift car...,spam,spam,spam,spam,spam,spam
2,Hot singles in your area are waiting to meet y...,work,spam,unknown,spam,work,spam
3,Re: Inheritance transfer of 4.5M USD pending y...,unknown,spam,unknown,spam,work,spam
4,Meeting agenda for Thursday's project review,unknown,work,unknown,work,unknown,work
5,"Q3 budget report attached, please review by Fr...",spam,work,spam,work,spam,work
6,Reminder: Annual performance review scheduled ...,spam,work,work,work,spam,work
7,"Updated draft of the manuscript, comments welcome",spam,work,spam,work,spam,work
8,Quick question about last week,spam,unknown,spam,unknown,spam,unknown
9,Following up,unknown,unknown,unknown,unknown,unknown,unknown


### *Comparison*
So, the 270m version was very unreliable and inconsistent. Which I am a bit surprised by, the prompt is quite clear and it shouldn't be this difficult to classify these. The 4b model performed in the way I expected a model to perform as it was correct and consistent. Understandably 4b was also slightly slower in my experience.

# **Task 2**

In [33]:
import json

df = pd.read_csv("https://raw.githubusercontent.com/sippohippo/PythonForBusinessInt2026/refs/heads/main/assignments/data/news.csv", sep=";")

#a Classification function with JSON
def classify_news(headline):
    headline_lower = headline.lower()

    topic_keywords = {
        "earnings":       ["earnings", "revenue", "profit", "forecast", "estimates", "quarterly"],
        "mergers":        ["acquire", "merger", "takeover", "deal", "talks"],
        "regulation":     ["regulators", "compliance", "licence", "rules", "block", "act", "fsa"],
        "macroeconomics": ["inflation", "interest rates", "borrowing", "eurozone", "financing"]
    }

    positive_keywords = ["beats", "surges", "steady", "grants", "cools", "easing", "expanded"]
    negative_keywords = ["misses", "weak", "block", "raise costs", "weigh", "climb"]

    topic = "unknown"
    for t, keywords in topic_keywords.items():
        if any(word in headline_lower for word in keywords):
            topic = t
            break

    if any(word in headline_lower for word in positive_keywords):
        sentiment = "positive"
    elif any(word in headline_lower for word in negative_keywords):
        sentiment = "negative"
    else:
        sentiment = "neutral"

    return json.dumps({"topic": topic, "sentiment": sentiment})

df["classification"] = df["headline"].apply(classify_news)

display(df)

,headline,classification
0,Nordion Industries beats Q1 earnings estimates...,"{""topic"": ""earnings"", ""sentiment"": ""positive""}"
1,Helvora Pharmaceuticals misses earnings foreca...,"{""topic"": ""earnings"", ""sentiment"": ""negative""}"
2,"Aurelis Bank reports steady quarterly profit, ...","{""topic"": ""earnings"", ""sentiment"": ""positive""}"
3,Veridyne Logistics to acquire rival Trantec in...,"{""topic"": ""mergers"", ""sentiment"": ""neutral""}"
4,Antitrust regulators block proposed merger bet...,"{""topic"": ""mergers"", ""sentiment"": ""negative""}"
5,Kestrel Semiconductor confirms early-stage mer...,"{""topic"": ""mergers"", ""sentiment"": ""neutral""}"
6,New EU AI Act compliance rules expected to rai...,"{""topic"": ""regulation"", ""sentiment"": ""negative""}"
7,Finnish FSA grants Norvik Capital expanded lic...,"{""topic"": ""regulation"", ""sentiment"": ""positive""}"
8,"Eurozone inflation cools to 2.1%, easing press...","{""topic"": ""macroeconomics"", ""sentiment"": ""posi..."
9,Rising interest rates weigh on Tessaro Real Es...,"{""topic"": ""macroeconomics"", ""sentiment"": ""nega..."


In [36]:
#b Classify with gemma3:4b
def classify_news_llm(headline, model):
    response = ollama.chat(model=model, messages=[
        {"role": "user", "content": (
            f"Classify this financial news headline by topic and sentiment.\n"
            f"Topic must be exactly one of: earnings, mergers, regulation, macroeconomics\n"
            f"Sentiment must be exactly one of: positive, negative, neutral\n"
            f"Reply only with a JSON object in this exact format, nothing else:\n"
            f"{{\"topic\": \"...\", \"sentiment\": \"...\"}}\n\n"
            f"Headline: {headline}"
        )}
    ])
    return response["message"]["content"].strip()

raw_results = df["headline"].apply(lambda x: classify_news_llm(x, "gemma3:4b"))

df["topic"]     = raw_results.apply(lambda x: json.loads(x)["topic"])
df["sentiment"] = raw_results.apply(lambda x: json.loads(x)["sentiment"])

display(df[["headline", "topic", "sentiment"]])

,headline,topic,sentiment
0,Nordion Industries beats Q1 earnings estimates...,earnings,positive
1,Helvora Pharmaceuticals misses earnings foreca...,earnings,negative
2,"Aurelis Bank reports steady quarterly profit, ...",earnings,positive
3,Veridyne Logistics to acquire rival Trantec in...,mergers,positive
4,Antitrust regulators block proposed merger bet...,regulation,negative
5,Kestrel Semiconductor confirms early-stage mer...,mergers,neutral
6,New EU AI Act compliance rules expected to rai...,regulation,negative
7,Finnish FSA grants Norvik Capital expanded lic...,regulation,positive
8,"Eurozone inflation cools to 2.1%, easing press...",macroeconomics,positive
9,Rising interest rates weigh on Tessaro Real Es...,macroeconomics,negative


### *Subtask c (Claude)*

| Headline | Topic | Sentiment |
|----------|-------|-----------|
| Nordion Industries beats Q1 earnings estimates as cloud revenue surges 28% | earnings | positive |
| Helvora Pharmaceuticals misses earnings forecast amid weak generics demand | earnings | negative |
| Aurelis Bank reports steady quarterly profit, in line with analyst expectations | earnings | neutral |
| Veridyne Logistics to acquire rival Trantec in 4.2 billion euro deal | mergers | neutral |
| Antitrust regulators block proposed merger between Solenta and Marvex Energy | mergers | negative |
| Kestrel Semiconductor confirms early-stage merger talks with Aldenfeld AG | mergers | neutral |
| New EU AI Act compliance rules expected to raise costs for Lumavex by 12% | regulation | negative |
| Finnish FSA grants Norvik Capital expanded licence for cross-border operations | regulation | positive |
| Eurozone inflation cools to 2.1%, easing pressure on Drava Holdings borrowing costs | macroeconomics | positive |
| Rising interest rates weigh on Tessaro Real Estate as financing costs climb | macroeconomics | negative |

Here Claude is a bit more inaccurate than Gemma3:4b regarding the sentiment, but more accurate regarding the topic (I would say headline 4 can be rightfully classified as both regulation/mergers). I can notice that Claude is more wiling to assign a neutral sentiment to the headlines, while Gemma only classifies one as neutral. But overall by the tiniest of margins, Gemma takes the cake i think since the only divergence from a) i can see with Gemma is changing one positive to neutral and one merger to regulation. Both were pretty good at this, the differences in answers can be written up as ambiguity in the headlines. I assume the reason Gemma3:4b outperformed Claude by this small bit is because it is maybe an overall more robust model than Sonnet 4.6.

# **Task 3**

In [38]:
!pip install ucimlrepo

from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
bank_marketing = fetch_ucirepo(id=222) 
  
# data (as pandas dataframes) 
X = bank_marketing.data.features 
y = bank_marketing.data.targets 
  
# variable information 
display(bank_marketing.variables)

,name,role,type,demographic,description,units,missing_values
0,age,Feature,Integer,Age,None,None,no
1,job,Feature,Categorical,Occupation,"type of job (categorical: 'admin.','blue-colla...",None,no
2,marital,Feature,Categorical,Marital Status,"marital status (categorical: 'divorced','marri...",None,no
3,education,Feature,Categorical,Education Level,"(categorical: 'basic.4y','basic.6y','basic.9y'...",None,no
4,default,Feature,Binary,None,has credit in default?,None,no
5,balance,Feature,Integer,None,average yearly balance,euros,no
6,housing,Feature,Binary,None,has housing loan?,None,no
7,loan,Feature,Binary,None,has personal loan?,None,no
8,contact,Feature,Categorical,None,contact communication type (categorical: 'cell...,None,yes
9,day_of_week,Feature,Date,None,last contact day of the week,None,no
